# TraceLine: learned collision classifier

Trains a small classifier on the **same pair features the rule engine uses** (`app.services.collision.FEATURES`), so the learned model and
the rules are scored on identical inputs with **leave-one-video-out** evaluation. Nothing here is validated for casework: a number from a
handful of clips is a smoke test. Report it with the number of videos and windows beside it.

You provide labelled clips in Google Drive:
- `traceline_data/videos/*.mp4`
- `traceline_data/labels.csv` with columns `video,kind,start_s,end_s`. One row per collision window (a video with two crashes has two rows). A video with **no** collision gets one row with empty `start_s,end_s` (negatives matter as much as positives). `kind` is `dashcam` or `cctv`.

Good public sources (check each licence before use): the ACCIDENT benchmark clips, CADP, DoTA, Nexar collision prediction. Include plenty of
**dense normal traffic** (queues, merging, overtaking) as negatives; that is exactly where false alarms come from.

In [ ]:
!git clone -q -b colab-ml-model https://github.com/sanjivinsmoke95/traceline.git
%cd traceline/backend
!pip -q install ultralytics scikit-learn joblib opencv-python-headless

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA = '/content/drive/MyDrive/traceline_data'
import pandas as pd, pathlib
labels = pd.read_csv(f'{DATA}/labels.csv')
videos = sorted(labels.video.unique())
print(len(videos), 'videos;', int(labels.start_s.notna().sum()), 'collision windows;', int(labels.groupby('video').start_s.apply(lambda s: s.isna().all()).sum()), 'negative videos')
labels.head()

## 1. Detect, track and build candidate pairs
Uses Ultralytics tracking (ByteTrack) at about 5 sampled frames per second, like the pipeline. Optional: set `CUE_WEIGHTS` to the accident-cue model's weights to include the cue feature (leave `None` to train without it; the feature is then -1 everywhere).

In [ ]:
import sys; sys.path.insert(0, '.')
from scripts.train_collision_classifier import tracks_from_video, candidates, leave_one_video_out, summarize, fit, pick_threshold, label
from app.services import collision
WEIGHTS = 'yolo11s.pt'   # any COCO detector; the production pipeline pins yolo26s
CUE_WEIGHTS = None       # e.g. '/content/drive/MyDrive/traceline_data/accident_yolov8s_best.pt'

data = {}
for v in videos:
    rows = labels[labels.video == v]
    wins = [[float(a), float(b)] for a, b in zip(rows.start_s, rows.end_s) if pd.notna(a) and pd.notna(b)]
    tracks, period, dur = tracks_from_video(f'{DATA}/videos/{v}', weights=WEIGHTS)
    data[v] = {'windows': wins, 'cands': candidates(tracks, kind=rows.kind.iloc[0], period_ms=period, duration_ms=dur)}
    print(f"{v}: {len(tracks)} tracks, {len(data[v]['cands'])} touching pairs, {len(wins)} windows")

## 2. Leave-one-video-out: learned model vs the rules
Each video is scored by a model that never saw it. The rules are scored the same way on the same videos.

In [ ]:
rows, rules = leave_one_video_out(data)
pd.DataFrame(rows)

In [ ]:
print('learned', summarize(rows))
print('rules  ', summarize(rules))
# windows = labelled collisions; found = windows with a flag; true/false_groups = flagged moments (within 2 s merged) inside / outside a window

## 3. Train on everything and export
Only do this after the evaluation above looks sensible *on videos you did not tune on*. The exported model is a development artefact until it has been checked on a held-out set.

In [ ]:
import joblib, json
X = [c['x'] for v in data for c in data[v]['cands']]
y = [label(c, data[v]['windows']) for v in data for c in data[v]['cands']]
m = fit(X, y); thr = pick_threshold(m, data, list(data))
out = f'{DATA}/collision_clf'; pathlib.Path(out).mkdir(exist_ok=True)
joblib.dump(m, f'{out}/collision_clf.joblib')
json.dump({'features': collision.FEATURES, 'threshold': thr, 'videos': len(data), 'positives': int(sum(y)), 'candidates': len(y),
           'note': 'development model, not validated for casework'}, open(f'{out}/collision_clf.json', 'w'), indent=1)
pd.Series(m.feature_importances_, index=collision.FEATURES).sort_values(ascending=False)

## Using it in TraceLine
The backend does **not** load this model yet. To use it: copy `collision_clf.joblib` and `.json` next to the weights, and have
`video_events.generate` call it on each touching candidate (`collision.features(ev, max_iou)` → `predict_proba` ≥ threshold) in place of or
alongside `collision.is_collision`. Keep the rules as the fallback and keep every flagged event carrying its `collision_signals`, so an
investigator can see *why*. Do that only once the leave-one-video-out numbers hold on clips that were never used to choose thresholds.